# Site Filtering: High Flood Risk & Data Quality

Goal: Filter Missouri Basin sites to keep only those with:
1. High streamflow variation (flood risk)
2. Good data quality (low null rates)

This analysis determines which sites to keep in the main `flood_model` table.

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb
import numpy as np

In [2]:
# Load the flood model dataset from wandb
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset:latest")
artifact_dir = artifact.download()

df = pl.read_parquet(f"{artifact_dir}/flood_model.parquet")
print(f"Full dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total sites: {df['site_id'].n_unique()}")
print(f"Date range: {df['observation_hour'].min()} to {df['observation_hour'].max()}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\sacha\_netrc.
wandb: Downloading large artifact 'flood-dataset:latest', 4522.32MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.4 (11566.0MB/s)


Full dataset: 101,651,130 rows x 51 columns
Total sites: 1029
Date range: 2007-10-28 05:00:00+00:00 to 2026-02-03 18:00:00+00:00


## 1. Analyze Streamflow Variation Per Site

We compute the coefficient of variation (CV) of streamflow for each site.
CV = std / mean. Higher CV means more variable flow, i.e., higher flood risk.

In [3]:
# Compute streamflow statistics per site
site_stats = df.group_by("site_id").agg(
    pl.col("streamflow_cfs_mean").mean().alias("streamflow_mean"),
    pl.col("streamflow_cfs_mean").std().alias("streamflow_std"),
    pl.col("streamflow_cfs_mean").max().alias("streamflow_max"),
    pl.col("streamflow_cfs_mean").min().alias("streamflow_min"),
    pl.col("gage_height_ft_mean").mean().alias("gage_height_mean"),
    pl.col("gage_height_ft_mean").std().alias("gage_height_std"),
    pl.col("latitude").first(),
    pl.col("longitude").first(),
    pl.col("station_name").first(),
    pl.len().alias("total_rows"),
    pl.col("streamflow_cfs_mean").null_count().alias("streamflow_nulls"),
    pl.col("gage_height_ft_mean").null_count().alias("gage_height_nulls"),
    pl.col("precipitation_mm").null_count().alias("precip_nulls"),
    pl.col("temperature_c").null_count().alias("temp_nulls"),
)

# Compute coefficient of variation and null rates
site_stats = site_stats.with_columns(
    (pl.col("streamflow_std") / pl.col("streamflow_mean")).alias("streamflow_cv"),
    (pl.col("streamflow_max") - pl.col("streamflow_min")).alias("streamflow_range"),
    (pl.col("streamflow_nulls") / pl.col("total_rows") * 100).alias("streamflow_null_pct"),
    (pl.col("gage_height_nulls") / pl.col("total_rows") * 100).alias("gage_height_null_pct"),
    (pl.col("precip_nulls") / pl.col("total_rows") * 100).alias("precip_null_pct"),
    (pl.col("temp_nulls") / pl.col("total_rows") * 100).alias("temp_null_pct"),
)

print(f"Total sites: {len(site_stats)}")
print(f"Sites with streamflow data (non-null CV): {site_stats.filter(pl.col('streamflow_cv').is_not_null()).shape[0]}")
print(f"Sites with 100% null streamflow (no data): {site_stats.filter(pl.col('streamflow_null_pct') == 100).shape[0]}")
print(f"Sites with zero mean streamflow: {site_stats.filter(pl.col('streamflow_mean') == 0).shape[0]}")

# Remove sites with no data or zero mean (zero mean causes inf CV)
site_stats = site_stats.filter(
    pl.col("streamflow_cv").is_not_null()
    & pl.col("streamflow_cv").is_finite()
    & (pl.col("streamflow_mean") > 0)
)
print(f"Sites after removing no-data and zero-mean: {len(site_stats)}")

# Show top 20 sites by CV
print("\nTop 20 sites by streamflow variation (highest flood risk):")
site_stats.select(
    "site_id", "station_name", "streamflow_cv", "streamflow_mean",
    "streamflow_range", "total_rows", "streamflow_null_pct", "gage_height_null_pct"
).sort("streamflow_cv", descending=True).head(20)

Total sites: 1029
Sites with streamflow data (non-null CV): 945
Sites with 100% null streamflow (no data): 84
Sites with zero mean streamflow: 2
Sites after removing no-data and zero-mean: 499

Top 20 sites by streamflow variation (highest flood risk):


site_id,station_name,streamflow_cv,streamflow_mean,streamflow_range,total_rows,streamflow_null_pct,gage_height_null_pct
str,str,f64,f64,f64,u32,f64,f64
"""06900050""","""Medicine Creek near Laredo, MO""",9287.315345,1.932855,1.034724e6,143176,1.211097,0.539895
"""06774000""","""Platte River near Duncan, Nebr…",1081.182031,42.972442,1.020199e6,144478,18.511469,0.481042
"""06482000""","""BIG SIOUX RIVER AT SIOUX FALLS…",308.933401,56.451979,1009841.5,112002,11.820325,54.882949
"""06746095""","""JOE WRIGHT CREEK ABOVE JOE WRI…",259.928538,10.383658,666882.436667,61063,0.067144,81.225947
"""06752260""","""CACHE LA POUDRE RIVER AT FORT …",209.02535,59.886173,1007971.5,137580,0.563309,100.0
…,…,…,…,…,…,…,…
"""06886500""","""FANCY C AT WINKLER, KS""",103.855411,39.987572,1.011574e6,64241,9.171713,1.084977
"""06607500""","""Little Sioux River near Turin,…",100.803068,462.49146,1.058049e6,144073,18.574611,0.038869
"""06483950""","""BIG SIOUX RIVER NEAR HAWARDEN,…",82.531637,668.067932,1.163499e6,60742,2.020019,37.410688


In [9]:
# Distribution of streamflow CV
cv_data = site_stats.to_pandas()
median_cv = cv_data["streamflow_cv"].median()

# Remove extreme outliers from plot data so bins spread evenly
p95 = cv_data["streamflow_cv"].quantile(0.95)
cv_clipped = cv_data[cv_data["streamflow_cv"] <= p95].copy()
n_outliers = len(cv_data) - len(cv_clipped)

fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3],
    subplot_titles=(
        f"Histogram of Streamflow CV ({len(cv_data)} sites, {n_outliers} outliers clipped)",
        "Box Plot (full range, shows outliers)"
    ))

fig.add_trace(
    go.Histogram(x=cv_clipped["streamflow_cv"], nbinsx=30,
                 name="Sites", marker_color="steelblue"),
    row=1, col=1
)
fig.add_vline(x=median_cv, line_dash="dash", line_color="red",
              annotation_text=f"Median: {median_cv:.2f}", row=1, col=1)

fig.add_trace(
    go.Box(x=cv_data["streamflow_cv"], name="CV", marker_color="steelblue"),
    row=2, col=1
)

fig.update_layout(height=500, showlegend=False, bargap=0.05)
fig.update_xaxes(title_text="CV (std/mean)", row=1, col=1)
fig.update_yaxes(title_text="Number of Sites", row=1, col=1)
fig.show()

print(f"CV range: {cv_data['streamflow_cv'].min():.2f} to {cv_data['streamflow_cv'].max():.2f}")
print(f"Median CV: {median_cv:.2f}")
print(f"Sites above median: {len(cv_data[cv_data['streamflow_cv'] > median_cv])}")
print(f"Sites below median: {len(cv_data[cv_data['streamflow_cv'] <= median_cv])}")
print(f"Outliers clipped from histogram (CV > {p95:.2f}): {n_outliers}")

CV range: 0.06 to 9287.32
Median CV: 2.03
Sites above median: 249
Sites below median: 250
Outliers clipped from histogram (CV > 41.86): 25


## 2. Analyze Null Rates Per Site

In [10]:
# Distribution of null rates
null_data = site_stats.to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Streamflow Null %", "Gage Height Null %"))

fig.add_trace(
    go.Histogram(x=null_data["streamflow_null_pct"], nbinsx=50, name="Streamflow"),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=null_data["gage_height_null_pct"], nbinsx=50, name="Gage Height"),
    row=1, col=2
)
fig.update_layout(title="Distribution of Null Rates Across Sites", showlegend=False)
fig.show()

# Summary
print(f"Sites with 0% streamflow nulls:    {len(null_data[null_data['streamflow_null_pct'] == 0])}")
print(f"Sites with <20% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] < 20])}")
print(f"Sites with >50% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] > 50])}")
print(f"Sites with 100% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] == 100])}")
print()
print(f"Sites with 0% gage height nulls:   {len(null_data[null_data['gage_height_null_pct'] == 0])}")
print(f"Sites with <20% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] < 20])}")
print(f"Sites with >50% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] > 50])}")
print(f"Sites with 100% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] == 100])}")

Sites with 0% streamflow nulls:    151
Sites with <20% streamflow nulls:  471
Sites with >50% streamflow nulls:  4
Sites with 100% streamflow nulls:  0

Sites with 0% gage height nulls:   26
Sites with <20% gage height nulls: 307
Sites with >50% gage height nulls: 183
Sites with 100% gage height nulls: 121


## 3. Apply Filters

Criteria:
- Streamflow CV > median (above-average variation = higher flood risk)
- Gage height null rate < 20% (good data quality)
- Streamflow null rate < 20% (good data quality)

Adjust thresholds based on the distributions above.

In [11]:
# Set thresholds (adjust after reviewing distributions above)
CV_THRESHOLD = site_stats.filter(
    pl.col("streamflow_cv").is_not_null()
)["streamflow_cv"].median()
NULL_THRESHOLD = 20  # max % nulls allowed

print(f"CV threshold (median): {CV_THRESHOLD:.2f}")
print(f"Null threshold: {NULL_THRESHOLD}%")

# Apply filters
filtered_sites = site_stats.filter(
    (pl.col("streamflow_cv") > CV_THRESHOLD)
    & (pl.col("streamflow_null_pct") < NULL_THRESHOLD)
    & (pl.col("gage_height_null_pct") < NULL_THRESHOLD)
)

print(f"\nBefore filtering: {len(site_stats)} sites")
print(f"After filtering:  {len(filtered_sites)} sites")
print(f"Removed:          {len(site_stats) - len(filtered_sites)} sites")

# Breakdown of why sites were removed
no_data = site_stats.filter(pl.col("streamflow_cv").is_null())
low_cv = site_stats.filter(
    (pl.col("streamflow_cv").is_not_null()) & (pl.col("streamflow_cv") <= CV_THRESHOLD)
)
high_nulls = site_stats.filter(
    (pl.col("streamflow_cv") > CV_THRESHOLD)
    & ((pl.col("streamflow_null_pct") >= NULL_THRESHOLD) | (pl.col("gage_height_null_pct") >= NULL_THRESHOLD))
)
print(f"\nRemoval breakdown:")
print(f"  No streamflow data at all: {len(no_data)}")
print(f"  Low variation (CV <= {CV_THRESHOLD:.2f}): {len(low_cv)}")
print(f"  High nulls (>= {NULL_THRESHOLD}%): {len(high_nulls)}")

CV threshold (median): 2.03
Null threshold: 20%

Before filtering: 499 sites
After filtering:  161 sites
Removed:          338 sites

Removal breakdown:
  No streamflow data at all: 0
  Low variation (CV <= 2.03): 250
  High nulls (>= 20%): 88


## 4. Visualizations: Before vs After

In [ ]:
# Map: All sites vs Filtered sites
all_sites_pd = site_stats.to_pandas()
all_sites_pd["status"] = "Removed"
kept_ids = set(filtered_sites["site_id"].to_list())
all_sites_pd.loc[all_sites_pd["site_id"].isin(kept_ids), "status"] = "Kept"

fig = px.scatter_geo(
    all_sites_pd,
    lat="latitude", lon="longitude",
    color="status",
    color_discrete_map={"Kept": "blue", "Removed": "red"},
    hover_name="station_name",
    hover_data=["site_id", "streamflow_cv", "streamflow_null_pct", "gage_height_null_pct"],
    title=f"Site Filtering: {len(filtered_sites)} Kept (blue) vs {len(site_stats) - len(filtered_sites)} Removed (red)",
    scope="usa",
)
fig.update_layout(geo=dict(center=dict(lat=45, lon=-105), projection_scale=3))
fig.show()

In [ ]:
# Scatter: CV vs Null Rate (shows filtering logic)
# Only show sites that have streamflow data (non-null CV)
scatter_data = all_sites_pd.copy()

fig = px.scatter(
    scatter_data,
    x="streamflow_cv", y="gage_height_null_pct",
    color="status",
    color_discrete_map={"Kept": "blue", "Removed": "red"},
    hover_name="station_name",
    hover_data=["site_id"],
    title="Streamflow Variation vs Data Quality (sites with data only)",
    labels={"streamflow_cv": "Streamflow CV (higher = more variable)",
            "gage_height_null_pct": "Gage Height Null %"},
)
fig.add_vline(x=CV_THRESHOLD, line_dash="dash", line_color="gray",
              annotation_text=f"CV threshold: {CV_THRESHOLD:.2f}")
fig.add_hline(y=NULL_THRESHOLD, line_dash="dash", line_color="gray",
              annotation_text=f"Null threshold: {NULL_THRESHOLD}%")
fig.show()

In [19]:
# Comparison: dataset size before vs after
kept_df = df.filter(pl.col("site_id").is_in(filtered_sites["site_id"].to_list()))

print(f"Rows before: {len(df):,}")
print(f"Rows after:  {len(kept_df):,}")
print(f"Reduction:   {(1 - len(kept_df)/len(df))*100:.1f}%")
print(f"\nSites before: {df['site_id'].n_unique()}")
print(f"Sites after:  {kept_df['site_id'].n_unique()}")

Rows before: 101,651,130
Rows after:  28,764,496
Reduction:   71.7%

Sites before: 1029
Sites after:  267


In [20]:
# Export the list of kept site IDs as a dbt seed CSV
from pathlib import Path

kept_site_ids = filtered_sites.select("site_id").sort("site_id")
print(f"Kept {len(kept_site_ids)} site IDs")

seed_path = Path("../elt/transformation/seeds/filtered_site_ids.csv")
kept_site_ids.write_csv(seed_path)
print(f"Saved to {seed_path.resolve()}")
print(kept_site_ids)

Kept 267 site IDs:
shape: (267, 1)
┌─────────────────┐
│ site_id         │
│ ---             │
│ str             │
╞═════════════════╡
│ 05018000        │
│ 06020600        │
│ 06023500        │
│ 06023800        │
│ 06024020        │
│ …               │
│ 411429095583801 │
│ 411450095582201 │
│ 423323099520101 │
│ 423552100085501 │
│ 463138110580001 │
└─────────────────┘
